# Building an Agent, Step by Step
**LangChain v1 + LangGraph + LangSmith**

Each stage fixes one limitation of the stage before it.

## Stage 0: Setup

In [19]:
# pip install -U langchain langgraph langchain-openai python-dotenv numpy
from dotenv import load_dotenv
load_dotenv(override=True)   # loads OPENAI_API_KEY and LANGSMITH_* settings from .env

MODEL = "openai:gpt-4o-mini"                  # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## Stage 1: Talk to an LLM
Type `quit` to stop. Tell it your name, then ask it back.

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL)

while True:
    user = input("You: ")
    if user == "quit": break
    print("You:", user)
    print("AI:", llm.invoke(user).content)

You: hi
AI: Hello! How can I assist you today?
You: my name is afnan
AI: Nice to meet you, Afnan! How can I assist you today?
You: what is my name
AI: I don’t know your name. However, if you tell me, I’ll be happy to use it in our conversation!


## Stage 2: Short-term memory
Memory is just *sending the history back every time*. First by hand:

In [5]:
history = []

while True:
    user = input("You: ")
    if user == "quit": break
    print("You:", user)
    history.append({"role": "user", "content": user})
    reply = llm.invoke(history)
    history.append({"role": "assistant", "content": reply.content})
    print("AI:", reply.content)

You: my name is afnan
AI: Nice to meet you, Afnan! How can I assist you today?
You: what day is it today
AI: Today is October 7, 2023. How can I help you further?
You: what is my name?
AI: Your name is Afnan. How can I assist you today, Afnan?


In [4]:
history

[{'role': 'user', 'content': 'hi my name is afnan'},
 AIMessage(content='Hi Afnan! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 13, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5632fb73e4', 'id': 'chatcmpl-ElUi8hA77nvNVSOpA4RxMLUW8tqd8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1eda1-f0fa-7340-b735-fc6b25cb6e2f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 12, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'ou

Now let the framework handle it. A **checkpointer** saves the conversation per `thread_id`, so we never touch the list.

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def ask(agent, text, thread="1"):
    result = agent.invoke({"messages": [{"role": "user", "content": text}]},
                          {"configurable": {"thread_id": thread}})
    return result["messages"][-1].content

def chat(agent, thread):
    while True:
        user = input("You: ")
        if user == "quit": break
        print("You:", user)
        assistant_response = ask(agent, user, thread)
        print("AI:", assistant_response)

agent = create_agent(MODEL, checkpointer=InMemorySaver())
chat(agent, thread="1") 

You: my name is afnan
AI: Nice to meet you, Afnan! How can I assist you today?
You: what is my name
AI: Your name is Afnan! How can I help you further?


In [13]:
chat(agent, thread="2")    # new thread = fresh conversation, it won't know you here

You: when will the gta 6 release
AI: As of my last update in October 2023, Rockstar Games had not officially announced a specific release date for Grand Theft Auto VI (GTA 6). However, there have been various rumors and speculation about its potential launch. For the most current information, I recommend checking official Rockstar Games announcements or their website.


💡 From here on, `chat(agent)` works for every stage if you'd rather type live than run the scripted examples.

## Stage 3: RAG — give it *our* knowledge
It doesn't know university policy. Load the handbook, chunk it, embed it, and give the agent a **search tool**.

In [16]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

docs = [Document(p.read_text(), metadata={"source": p.name}) for p in Path("docs").glob("*.md")]
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings())

@tool
def search_handbook(query: str) -> str:
    """Search the university handbook for policies: exams, attendance, grading, fees."""
    return "\n\n".join(d.page_content for d in vector_store.similarity_search(query, k=3))

In [17]:
agent = create_agent(MODEL, tools=[search_handbook], checkpointer=InMemorySaver(),
                     system_prompt="You are a university assistant. Use the handbook for policy questions.")
ask(agent, "What's the minimum attendance to sit the final exam?")

ImportError: cosine_similarity requires numpy to be installed. Please install numpy with `pip install numpy`.

## Stage 4: Tools — from chatbot to agent
It can *answer*, but it can't *do* anything. Tools are plain functions; the **docstring** is what the model reads.

In [ ]:
@tool
def check_timetable(course_code: str) -> str:
    """Get the class schedule for a course code like CS101."""
    return {"CS101": "Mon/Wed 10:00, Room B2", "AI301": "Tue/Thu 14:00, Lab 4"}.get(course_code, "Not found")

@tool
def book_study_room(room: str, time: str) -> str:
    """Book a library study room at a given time."""
    return f"✅ Booked {room} at {time}"

tools = [search_handbook, check_timetable, book_study_room]
agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver())
ask(agent, "When is AI301? Book me study room 3 an hour before it.")

👉 Open LangSmith and show the trace: model → tool call → tool result → model.

## Stage 5: Human-in-the-loop
Actions with consequences should ask first. One line of middleware.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver(),
                     middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_study_room": True})])

config = {"configurable": {"thread_id": "hitl"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Book room 3 at 5pm"}]}, config)
result["__interrupt__"]                    # ⏸️ paused, waiting for a human

In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)   # or "reject"
result["messages"][-1].content

## Stage 6: Subagents
Too many tools in one agent → bloated context and tool confusion. Split into **specialists**, and let a supervisor delegate to them. Each subagent becomes a tool.

In [ ]:
policy_agent   = create_agent(MODEL, tools=[search_handbook], system_prompt="You answer university policy questions.")
schedule_agent = create_agent(MODEL, tools=[check_timetable, book_study_room], system_prompt="You handle timetables and bookings.")

@tool
def ask_policy_expert(question: str) -> str:
    """Delegate any question about university rules and policies."""
    return policy_agent.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].content

@tool
def ask_scheduler(request: str) -> str:
    """Delegate anything about class times or booking rooms."""
    return schedule_agent.invoke({"messages": [{"role": "user", "content": request}]})["messages"][-1].content

supervisor = create_agent(MODEL, tools=[ask_policy_expert, ask_scheduler], checkpointer=InMemorySaver(),
                          system_prompt="You coordinate specialists. Delegate, then give one combined answer.")
ask(supervisor, "Can I retake CS101's exam if I fail? Also when is the class?")

Every agent is a **LangGraph graph** under the hood:

In [ ]:
from IPython.display import Image
Image(supervisor.get_graph().draw_mermaid_png())

## Stage 7: Long-term memory
The checkpointer only remembers *one thread*. A **store** remembers facts about the user *across all chats*.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

@tool
def save_memory(fact: str, runtime: ToolRuntime) -> str:
    """Save an important, lasting fact about the student (program, courses, preferences)."""
    runtime.store.put(("memories", "ali"), fact, {"fact": fact})
    return "Saved."

@tool
def recall_memories(runtime: ToolRuntime) -> str:
    """Recall everything we know about the student. Use at the start of a conversation."""
    return "\n".join(m.value["fact"] for m in runtime.store.search(("memories", "ali")))

agent = create_agent(MODEL, tools=[*tools, save_memory, recall_memories],
                     checkpointer=InMemorySaver(), store=InMemoryStore())
ask(agent, "I'm in BS AI, 5th semester, and I hate morning classes.", thread="monday")

In [ ]:
ask(agent, "Suggest a good time for me to book a study room.", thread="friday")   # 🧠 a brand new chat that still knows

## Stage 8: LangSmith
Open the project **uni-workshop** and walk through:
- the full trace tree of the supervisor → subagent → tool calls
- token usage, latency and cost per step
- the store and memory tool calls across threads